# AICIDENTIA - 05 RQ3: risk-domain classification and label reliability

**RQ3:** How accurately can incident text be classified into risk domains, and how reliable are the AI-generated (MIT) risk labels, judged against independent signals already in AIID?

> CRISP-DM phases: **Modeling** (classifiers) and **Evaluation** (nested cross-validation, bootstrap CIs, paired tests, calibration, kappa); **Deployment** feeds the Streamlit app

Logic lives in `aicidentia.models.classify`, `classify_outputs` and `aicidentia.evaluation.metrics`; the write-up is `docs/rq3_classification.md`. **The target (MIT Risk Domain) is machine-generated**: classifier accuracy shows how well text reproduces that labeller, not agreement with humans.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
pd.set_option("display.width", 220, "display.max_colwidth", 90)
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Evaluate every classifier

Folds are fixed once and saved for the Colab run; hyperparameters are tuned by an inner 3-fold CV on training data only (nested CV). Everything is cached in `data/interim/`, so this takes seconds after the first run.

In [ ]:
from aicidentia.models import classify_outputs

res = classify_outputs.run_classification()
classify_outputs.write_all(res)
f = res.facts
print(f["labelled"], "labelled incidents;", len(f["classes"]), "classes;", f["class_counts"])
print(
    "time split train/test:",
    f["time_train"],
    "/",
    f["time_test"],
    "| DistilBERT scored:",
    res.colab_present,
)

## 2. Results: macro-F1 with bootstrap confidence intervals

Macro-F1 is primary because the classes are very unequal (a majority-class model gets about 38% accuracy but a macro-F1 near 0.08).

In [ ]:
display(
    res.metrics[
        ["model", "setting", "n", "macro_f1", "ci_low", "ci_high", "weighted_f1", "accuracy"]
    ].round(3)
)
display(Image(filename=FIG / "fig_rq3_1_model_comparison.png"))

**Paired comparisons** (same incidents, paired bootstrap of the macro-F1 difference and exact McNemar test). Most differences between the real models are within noise.

In [ ]:
display(
    res.significance[
        ["model_a", "model_b", "setting", "diff", "ci_low", "ci_high", "p_value", "mcnemar_p"]
    ].round(4)
)
print("best cross-validated model:", res.best_model)
print("deployable:", res.deployable, "|", res.deployable_reason)

## 3. Which domains are hard?

In [ ]:
d = res.data
support = pd.Series({c: int((d.y == i).sum()) for i, c in enumerate(d.classes)})
pc = res.per_class[
    res.per_class["class"].isin(d.classes)
    & ~res.per_class["model"].str.contains("ablation|majority")
]
display(
    pc.pivot_table(index="class", columns="model", values="f1")
    .round(3)
    .reindex(support.sort_values(ascending=False).index)
)
display(Image(filename=FIG / "fig_rq3_3_per_class_f1.png"))
display(Image(filename=FIG / "fig_rq3_2_confusion.png"))
display(res.confusions_top.round(3))

## 4. Calibration of the confidence shown to users

Three options compared on out-of-fold predictions (calibrators fitted inside each training part): uncalibrated, Platt (sigmoid) and temperature scaling. Platt made things *worse* here; temperature scaling is selected by the pre-specified rule.

In [ ]:
cal = res.calibration
print(
    pd.DataFrame(
        {
            m: {
                k: round(cal[m][k], 4)
                for k in ["ece", "brier", "accuracy", "mean_confidence", "macro_f1"]
            }
            for m in ["raw", "sigmoid", "temperature"]
        }
    )
)
print("selected:", cal["chosen"])
display(Image(filename=FIG / "fig_rq3_4_reliability.png"))

## 5. Highest-weight terms per domain


In [ ]:
display(res.features[res.features["model"] == classify_outputs.BASELINE][["domain", "top_terms"]])

## 6. DistilBERT (local GPU; Colab as the alternative)

`scripts/train_distilbert_local.py` runs the same training code as `notebooks/colab_distilbert_finetune.ipynb` (identical folds, seeds and protocol) on the local NVIDIA GPU and writes `reports/colab/rq3_distilbert_predictions.csv`; the model is then scored in the tables above whether or not it beats the baseline.

In [ ]:
import json

colab = PROJECT_ROOT / "reports" / "colab"
print({p.name: p.stat().st_size for p in colab.glob("*.csv")})
print("DistilBERT predictions present:", res.colab_present)
run_log = colab / "rq3_distilbert_run.json"
if run_log.exists():
    print(json.loads(run_log.read_text(encoding="utf-8")))

## 7. How reliable are the AI-generated (MIT) labels?

No hand annotation (decision D1 in `docs/decisions.md`). Three complementary checks use independent signals already in AIID; they ran inside `run_classification()` above (`aicidentia.models.reliability`).

In [ ]:
rel = res.reliability
display(Image(filename=FIG / "fig_rq3_5_label_reliability.png"))

### (a) Agreement with human CSETv1 / GMF annotations

Partial, convergent-validity checks: each maps one MIT domain to the closest human field. Not ground truth.

In [ ]:
display(
    rel.mappings[
        [
            "mit_domain",
            "human_field",
            "n",
            "excluded_unclear",
            "agreement",
            "kappa",
            "kappa_low",
            "kappa_high",
            "band",
        ]
    ].round(3)
)

### (b) Consistency on editor-linked pairs

Same-MIT-domain rate on pairs AIID editors marked as similar, against random pairs matched on year, and on editor-marked dissimilar pairs.

In [ ]:
display(pd.DataFrame(rel.pairs).T.round(3))
display(rel.pair_domains.head(8))

### (c) Confident learning (Northcutt et al., 2021)

cleanlab on the best classifier's out-of-fold probabilities: estimated noise rate per domain, flagged incidents and the confusion structure.

In [ ]:
print(rel.cl.facts)
display(rel.cl.noise_by_domain.round(3))
display(Image(filename=FIG / "fig_rq3_6_label_confusion.png"))
display(rel.symmetric.round(1).head(8))
titles = res.data.frame.set_index("incident_id")["title"]
from aicidentia.models.reliability_outputs import flagged_examples

display(flagged_examples(rel, titles))

### Sensitivity: retrain without likely mislabelled incidents

Flags are recomputed inside each training part (nested), so no test label decides which incidents are removed. This checks the reliability finding; it does not replace the headline scores.

In [ ]:
display(rel.retrain.round(4))

## 8. Registry

In [ ]:
for key in [
    "rq3.best_model",
    "rq3.deployable",
    "rq3.calibration_method",
    "rq3rel.mapping_kappa",
    "rq3rel.cl_noise_rate",
    "rq3rel.retrain_diff",
]:
    print(key, "=", registry.get(key))
print("editor-similar pairs:", registry.get("rq3rel.pairs_similar"))